# Species Conditioning -- Training (3 runs)

Trains `SpeciesConditionedMTL` on seeds 42, 43, 44 using the existing multi-task loop with Equal Weighting. The only difference from D-EW is the model class: same split, same hyperparameters, same checkpoint criterion, same determinism settings.

Uses `02_Manifests/split_manifest.csv`, the group-level split -- not the image-level one from the leakage ablation.

Writes only under `09_Species_Conditioning/`; the main results and the ablation are untouched.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'

COND_DIR = '/content/drive/MyDrive/fish-freshness-mtl/species_conditioning'
CHECKPOINT_DIR = f'{COND_DIR}/checkpoints'
RESULTS_DIR = f'{COND_DIR}/results'

import os
for d in ['/content/data', CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
!unzip -q -n "$DATASET_ZIP" -d /content/data

Re-run the shape and gradient checks here too. A detached gradient is not visible in the loss curve, so if it were ever lost the training would look perfectly normal while measuring the wrong thing.

In [ ]:
import sys
sys.path.append('/content/repo/04_Src')

!python 04_Src/test_species_conditioning.py

In [ ]:
import pandas as pd
from models import SpeciesConditionedMTL
from split_utils import load_split, verify_split
from train import train_multitask, TrainConfig

SPLIT_PATH = '/content/repo/02_Manifests/split_manifest.csv'
verify_split(pd.read_csv(SPLIT_PATH))  # same group-level split as the main experiment
train_df, val_df, test_df = load_split(SPLIT_PATH)
len(train_df), len(val_df), len(test_df)

In [ ]:
import json
import matplotlib.pyplot as plt
from plotting import plot_multitask_curves

SEEDS = [42, 43, 44]
cfg = TrainConfig()  # unchanged from the main experiment

for seed in SEEDS:
    run_name = f'ModelD_Cond_seed{seed}'
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'{run_name}.pt')
    if os.path.exists(ckpt_path):
        print(f'[{run_name}] checkpoint exists, skipping')
        continue
    print(f'[{run_name}] starting...')

    result = train_multitask(
        'EW',
        train_df=train_df, val_df=val_df, dataset_root=DATASET_ROOT,
        checkpoint_path=ckpt_path, seed=seed, cfg=cfg, run_name=run_name,
        model_factory=SpeciesConditionedMTL,
    )

    with open(os.path.join(RESULTS_DIR, f'{run_name}_history.json'), 'w') as f:
        json.dump(result['history'], f)
    fig = plot_multitask_curves(result['history'], run_name,
                                save_path=os.path.join(RESULTS_DIR, f'{run_name}_curves.png'))
    plt.close(fig)

In [ ]:
print(len([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pt')]), 'checkpoints (expect 3)')